# NAS - Optuna

- **Authored by:** Matheus Ferreira Silva 
- **GitHub:**: https://github.com/MatheusFS-dev

## 1. Setup and Configuration

### 1.1. Environment Variables

In [1]:
import os

# Async CUDA allocator
os.environ['TF_GPU_ALLOCATOR'] = 'cuda_malloc_async'

# If cuDNN autotune fails, fall back to a safe (but slower) algorithm.
os.environ["XLA_FLAGS"] = "--xla_gpu_strict_conv_algorithm_picker=false"

# Allow TensorFlow to allocate GPU memory as needed
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true' 

In [2]:
# Disable all auto-JIT clustering at the process level
os.environ["TF_XLA_FLAGS"] = "--tf_xla_auto_jit=-1"

### 1.2. Imports

In [3]:
from _imports import * # Centralized file containing all imports

2025-07-11 08:56:18.765212: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-07-11 08:56:18.779802: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1752234978.796861 2180601 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1752234978.802286 2180601 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-07-11 08:56:18.820271: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [4]:
# Disable all XLA auto-JIT compilation
tf.config.optimizer.set_jit(False)

# Spektral’s GCNConv uses a sparse-dense matmul under the hood.
# XLA’s GPU JIT compiler does not support that op.

### 1.3. GPU Management

In [5]:
# Specify GPU to use (e.g., GPU 0)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
get_gpu_info()


TensorFlow GPU Monitor - 2025-07-11 08:56:20
TensorFlow Configuration
Version        : 2.18.0
CUDA Support   : Yes
CUDA Version   : 12.5.1
cuDNN Version  : 9

GPU Information
GPU Name                      Memory Usage         Temp   Util  
--------------------------------------------------------------------------------
0   NVIDIA GeForce RTX 3070      1.5GB /    8.0GB  47C    23%   



2025-07-11 08:56:20.826750: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1752234980.826772 2180601 gpu_process_state.cc:201] Using CUDA malloc Async allocator for GPU: 0
I0000 00:00:1752234980.827905 2180601 gpu_device.cc:2022] Created device /device:GPU:0 with 4756 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3070 Ti, pci bus id: 0000:b3:00.0, compute capability: 8.6


## 2. Run Parameters 

In [6]:
NUM_TRIALS = 500
EPOCHS = 100

SEED = 99

In [7]:
# Direction for optimization of the study value
DIRECTION = "minimize"  # "minimize" or "maximize"

In [8]:
POLICY = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(POLICY)

In [9]:
TOP_K = 10 # Number of top trials to save

# True -> the greatest, the better
# False -> the least, the better
RANK_DESCENDING = True  

# Key to rank trials by:
# "value" -> objective trial value
# other e.g., "test_accuracy" -> user params
RANK_KEY = "test_accuracy_s009_full"

In [10]:
# Set to an existing path to resume training
# Set to None to start a new run
RESUME_TRAINING_PATH = "runs/nas_cnn1d_flat_v21.0"
RUN_DIR = RESUME_TRAINING_PATH or create_run_directory(prefix="nas_")

## 3. Data Loading and Preprocessing

In [11]:
(
    s008_coord_input,
    s008_lidar_input,
    s008_y_train,
    s009_coord_input,
    s009_lidar_input,
    s009_y,
) = load_dataset_sparse_labels()

/home/matheus/src/RayWise/src/_load_dataset.py:42: ComplexWarning: Casting complex values to real discards the imaginary part
  s008_y_train = s008_y_train.astype(np.float32)


Shape before conversion: (9234, 8, 32)
Shape after conversion: (9234,)
y_train shape: (9234,)
coord_input shape: (9234, 2)
lidar_input shape: (9234, 20, 200, 10)


/home/matheus/src/RayWise/src/_load_dataset.py:65: ComplexWarning: Casting complex values to real discards the imaginary part
  s008_y_val = s008_y_val.astype(np.float32)


Shape before conversion: (1960, 8, 32)
Shape after conversion: (1960,)
y_val shape: (1960,)
coord_input_val shape: (1960, 2)
lidar_input_val shape: (1960, 20, 200, 10)
y_train shape: (11194,)
coord_input shape: (11194, 2)
lidar_input shape: (11194, 20, 200, 10)


/home/matheus/src/RayWise/src/_load_dataset.py:101: ComplexWarning: Casting complex values to real discards the imaginary part
  s009_y = s009_y.astype(np.float32)


Shape before conversion: (9638, 8, 32)
Shape after conversion: (9638,)
y shape: (9638,)
coord_input shape: (9638, 2)
lidar_input shape: (9638, 20, 200, 10)


In [12]:
(
    x_s008_lidar_train,
    x_s008_lidar_val,
    x_s008_coord_train,
    x_s008_coord_val,
    y_s008_train,
    y_val,
) = train_test_split(
    s008_lidar_input,
    s008_coord_input,
    s008_y_train,
    test_size=0.2,
    random_state=SEED,
    shuffle=True,
)

(
    x_s009_lidar_test,
    x_s009_lidar_val,
    x_s009_coord_test,
    x_s009_coord_val,
    y_s009_test,
    y_s009_val,
) = train_test_split(
    s009_lidar_input,
    s009_coord_input,
    s009_y,
    test_size=0.2,
    random_state=SEED,
    shuffle=True,
)

x_lidar_train = x_s008_lidar_train
x_coord_train = x_s008_coord_train
y_train = y_s008_train

x_lidar_val = np.concatenate((x_s008_lidar_val, x_s009_lidar_val), axis=0)
x_coord_val = np.concatenate((x_s008_coord_val, x_s009_coord_val), axis=0)
y_val = np.concatenate((y_val, y_s009_val), axis=0)

x_lidar_test = x_s009_lidar_test
x_coord_test = x_s009_coord_test
y_test = y_s009_test

## 4. Getters

### 4.1. Callbacks

In [13]:
def get_callbacks(trial: optuna.Trial, backup_dir: str) -> List[tf.keras.callbacks.Callback]:
    """
    Constructs and returns a list of Keras callbacks tailored for Optuna trials.

    Args:
        trial (optuna.Trial): The current Optuna trial object.
        backup_dir (str): Directory where the backup files will be stored.

    Returns:
        List[tf.keras.callbacks.Callback]: A list of callbacks to pass into `model.fit()`.
    """
    # Metric to monitor for early stopping and checkpointing
    monitor: str = "val_loss"

    # Stop training early if no improvement in validation loss for N epochs
    early_stopping = callbacks.EarlyStopping(
        monitor=monitor,
        patience=10,  # number of epochs to wait
        restore_best_weights=True,
        verbose=1,
    )

    # Reduce learning rate if validation loss plateaus
    reduce_lr = callbacks.ReduceLROnPlateau(
        monitor=monitor,
        patience=5,  # how many epochs to wait before reducing LR
        factor=0.2,  # reduce LR by this factor
        min_lr=1e-6,  # don't reduce below this
        verbose=1,
    )
    
    # Backup and restore the model
    # backup = callbacks.BackupAndRestore(backup_dir=backup_dir)
    
    # Model checkpointing
    # checkpoint = callbacks.ModelCheckpoint(
    #     filepath=os.path.join(backup_dir, "checkpoint.h5"),
    #     monitor=monitor,
    #     save_best_only=True,
    #     save_weights_only=True,
    # )

    #! ——————— WARNING: the callbacks below do not work with multi-objective —————— !#
    # Custom callback to prune trial if NaN loss is encountered
    nan_pruner_callback = callbacks.TerminateOnNaN()

    # Optuna's built-in pruning callback for early trial termination
    pruning_callback = KerasPruningCallback(trial, monitor, interval=5)
    #! ———————————————————————————————————————————————————————————————————————————— !#

    # Return the complete list of callbacks
    return [early_stopping, reduce_lr, nan_pruner_callback, pruning_callback]


def get_activation(function: str) -> tf.keras.layers.Layer:
    """
    Returns the activation layer based on the provided function name.
    
    Args:
        function (str): Name of the activation function.
        
    Returns:
        tf.keras.layers.Layer: Corresponding activation layer.
    """
    if function == "relu":
        return layers.Activation("relu")
    elif function == "tanh":
        return layers.Activation("tanh")
    elif function == "sigmoid":
        return layers.Activation("sigmoid")
    elif function == "swish":
        return layers.Activation("swish")
    else:
        raise ValueError(f"Unsupported activation function: {function}")

## 5. Hyperparameters

In [14]:
hparams = HParams(
    activation_choices=[
        "None",
        #? ReLU family
        "relu",
        #? Smooth ReLU-like and modern variants
        "swish",
        #? Tanh family
        "tanh",
    ],
    regularizer_choices=[
        "none",
        "l1",
        "l2",
        "l1l2",
    ],
    optimizer_choices=[
        "Lion",
    ],
    scaler_choices=[
        "StandardScaler",
        "MinMaxScaler_0_1",
        "MinMaxScaler_-1_1",
        "RobustScaler",
        "QuantileTransformer",
        "PowerTransformer",
    ],
    l1_value=1e-2,
    l2_value=1e-2,
    min_lr=3e-5,
    max_lr=8e-5,
)

initializer_options = [
    initializers.Zeros(),
    initializers.Ones(),
    initializers.Constant(),
    initializers.RandomNormal(),
    initializers.RandomUniform(),
    initializers.TruncatedNormal(),
    initializers.GlorotNormal(),
    initializers.GlorotUniform(),
    initializers.HeNormal(),
    initializers.HeUniform(),
    initializers.LecunNormal(),
    initializers.LecunUniform(),
    initializers.Identity(),
    initializers.Orthogonal(),
    initializers.VarianceScaling(),
]

## 6. Objective Function

In [15]:
def build_model(
    trial: optuna.Trial, hparams: dict, show_summary: bool = True, t_seed: int = 42
) -> tf.keras.Model:
    # ———————————————————————————————————————————————————————————————————————————— #
    #                              Model Construction                              #
    # ———————————————————————————————————————————————————————————————————————————— #

    # ———————————————————————————————— LiDAR Input ——————————————————————————————— #
    x_lidar_input = layers.Input(shape=(20, 200, 10), name="lidar_input")

    # Inline one-hot encoding of semantic values
    one_hot_lidar = layers.Lambda(
        lambda x: tf.concat(
            [
                # “Is there a BS anywhere in the 10 channels?” → 1 channel
                tf.cast(tf.reduce_any(tf.equal(x, -2), axis=-1, keepdims=True), tf.float32),
                # “Vehicle?” → 1 channel
                tf.cast(tf.reduce_any(tf.equal(x, -1), axis=-1, keepdims=True), tf.float32),
                # “Obstacle?” → 1 channel
                tf.cast(tf.reduce_any(tf.equal(x, 1), axis=-1, keepdims=True), tf.float32),
                # “Free?” → 1 channel (all channels zero)
                tf.cast(tf.reduce_all(tf.equal(x, 0), axis=-1, keepdims=True), tf.float32),
            ],
            axis=-1,
        ),
        #! Lambda has deserialization issues, so providing the output shape is necessary
        output_shape=(20, 200, 4),
        name="lidar_transform_to_one_hot",
    )(x_lidar_input)
    # -> (batch, 20, 200, 4)

    # Flatten the 20×200 grid into a 4000-length sequence with the 4 channels
    x_lidar_flat: layers.Layer = layers.Reshape((20 * 200, 4), name="lidar_flatten_4_channels")(one_hot_lidar)

    # ———————————————————————————————— GPS Input ———————————————————————————————— #
    # Input for coordinate data (e.g., shape: (2,))
    x_coord_input = layers.Input(shape=(2,), name="coord_input")

    # Turn (batch,2) → (batch,1,2) → tile to (batch,4000,2)'
    x_coord: layers.Layer = layers.Lambda(
        lambda x: tf.tile(tf.expand_dims(x, axis=1), [1, 20 * 200, 1]),
        #! Lambda has deserialization issues, so providing the output shape is necessary
        output_shape=(20 * 200, 2),
        name="coord_tile_flat",
    )(x_coord_input)

    # ————————————————————————————— Combine Branches ————————————————————————————— #
    # Fuse channels:  (batch,4000,4) + (batch,4000,2) → (batch,4000,6)
    combined = layers.Concatenate(axis=-1, name="combine_lidar_coord")([x_lidar_flat, x_coord])

    # ———————————————————————————————— Initializer ———————————————————————————————— #
    initializer = tf.keras.initializers.GlorotUniform(
        seed=t_seed,
    )

    # —————————————————————————————— CNN + SE Branch ————————————————————————————— #
    num_conv_layers = trial.suggest_int("num_conv_layers", 1, 4)

    for i in range(num_conv_layers):
        # First Conv1D layer
        x = build_cnn1d(
            trial=trial,
            hparams=hparams,
            x=combined if i == 0 else x,  # Use combined only for the first layer
            name_prefix=f"conv1d_{i}",
            # Filters
            filters_range=trial.suggest_categorical(f"conv1d_{i}_filters", [64, 128, 256, 512]),
            # filters_step=40,
            # Kernel size
            kernel_size_range=(2, 6),
            kernel_size_step=2,
            # Other parameters
            # strides=trial.suggest_int(f"conv1d_{i}_strides", 1, 2),
            kernel_initializer=initializer,
        )
        pool_size = trial.suggest_int(f"pool_size_{i}", 2, 8, step=1)
        x = layers.MaxPooling1D(pool_size=pool_size, name=f"max_pool_{i}")(x)
        x = build_squeeze_excite_1d(
            x=x,
            trial=trial,
            hparams=hparams,
            ratio_choices=[8, 16, 32, 64],
            name_prefix=f"se_{i}",
        )

    # —————————————————————————————— Flatten CNN Output ————————————————————————————— #
    # Flatten the CNN output to 2D
    x = layers.Flatten(name="flatten_cnn_output")(x)
    
    # ———————————————————————————— Extra dense layers ———————————————————————————— #
    num_dense_layers = trial.suggest_int("num_dense_layers", 0, 3)

    for i in range(num_dense_layers):
        # Dense layer with dropout
        x = build_dnn(
            trial=trial,
            hparams=hparams,
            x=x,
            name_prefix=f"dense_{i}",
            # Units
            units_range=(250, 600),
            units_step=50,
            # Dropout
            dropout_rate_range=(0.0, 0.4),
            dropout_rate_step=0.2,
            # Other parameters
            kernel_initializer=initializer,
        )

    # —————————————————————————————————— Output —————————————————————————————————— #
    outputs = layers.Dense(
        256,
        activation="softmax",
        name="output",
        kernel_initializer=initializer,
    )(x)

    # —————————————————————————— Set Inputs and Outputs —————————————————————————— #
    model = Model(inputs=(x_lidar_input, x_coord_input), outputs=(outputs,))

    # ———————————————————————————————————————————————————————————————————————————— #
    #                                Train the Model                               #
    # ———————————————————————————————————————————————————————————————————————————— #
    model.summary() if show_summary else None

    model.compile(
        optimizer=hparams.get_optimizer(trial),
        loss=losses.SparseCategoricalCrossentropy(),
        metrics=["accuracy"],
        jit_compile=False,  # Disable XLA JIT compilation
    )

    return model

In [16]:
def objective(
    trial: optuna.Trial,
    backup_dir: str,
    model_dir: str,
    fig_dir: str,
    logs_dir: str,
    history_dir: str,
    epochs: int = 50,
    size_penalizer: Optional[str] = None,
) -> float:
    """
    Objective function for Optuna to optimize a Neural Network NN on any-input data.

    Args:
        trial (optuna.Trial): Current trial for hyperparameter suggestions.
        X (List[np.ndarray]): List of input arrays.
        y (List[np.ndarray]): List of label arrays.
        backup_dir (str): Path to store backup files.
        model_dir (str): Path to store full models.
        fig_dir (str): Path to store plots.
        logs_dir (str): Path to store logs.
        history_dir (str): Path to store training history.
        epochs (int): Number of training epochs.
        size_penalizer (Optional[str]): type of penalizer to use:
            - "params": Penalizes based on the number of parameters.
            - "flops": Penalizes based on the number of FLOPs.
            - None: No penalization is applied.

    Returns:
        float: Final validation loss (optionally penalized) used for optimization.
    """
    (clear_session(), gc.collect())  # Redundancy cleanup

    # ——————————————————————————————————— Setup —————————————————————————————————— #
    global x_lidar_train
    global x_coord_train
    global y_train
    global x_lidar_val
    global x_coord_val
    global y_val
    global x_lidar_test
    global x_coord_test
    global y_test
    global s009_lidar_input
    global s009_coord_input
    global s009_y

    t_seed = 364

    # Each trial gets a different seed
    np.random.seed(t_seed)
    tf.random.set_seed(t_seed)

    # ———————————————————————————————————————————————————————————————————————————— #

    model = None
    try:

        # ———————————————————————————————————————————————————————————————————————————— #
        #                              Data Preprocessing                              #
        # ———————————————————————————————————————————————————————————————————————————— #
        coord_scaler = StandardScaler()

        coord_scaler.fit(x_coord_train)
        x_coord_train = coord_scaler.transform(x_coord_train)
        x_coord_val = coord_scaler.transform(x_coord_val)
        x_coord_test = coord_scaler.transform(x_coord_test)
        s009_coord_input = coord_scaler.transform(s009_coord_input)

        # ———————————————————————————————————————————————————————————————————————————— #
        #                              Model Construction                              #
        # ———————————————————————————————————————————————————————————————————————————— #
        model = build_model(trial=trial, hparams=hparams, show_summary=True, t_seed=t_seed)

        batch_size = 64
        history = model.fit(
            [x_lidar_train, x_coord_train],
            y_train,
            validation_data=([x_lidar_val, x_coord_val], y_val),
            epochs=epochs,
            batch_size=batch_size,
            callbacks=get_callbacks(trial, backup_dir),
            verbose=2,
        )

        model.save(os.path.join(model_dir, f"trial_{trial.number}.keras"))

        # ———————————————————————————————————————————————————————————————————————————— #
        #                            Penalize the Model Size                           #
        # ———————————————————————————————————————————————————————————————————————————— #
        loss_values = punish_model(
            target=history.history["val_loss"],
            model=model,
            type=size_penalizer,
            flops_penalty_factor=1e-10,
            params_penalty_factor=1e-9,
            direction=DIRECTION,
        )
        loss = min(loss_values)

        # ———————————————————————————————————————————————————————————————————————————— #
        #                              Save Trial Results                              #
        # ———————————————————————————————————————————————————————————————————————————— #

        # ———————————————————————— Save model characteristics ———————————————————————— #
        get_model_stats(
            trial=trial,
            model=model,
            bits_per_param=tf.dtypes.as_dtype(POLICY.variable_dtype).size,
            batch_size=batch_size,
            n_trials=1000,
        )

        # ————————————————————————————— Evaluate on s009 ————————————————————————————— #
        test_loss, test_acc = model.evaluate(
            [x_lidar_test, x_coord_test], y_test, batch_size=batch_size, verbose=0
        )

        trial.set_user_attr("test_accuracy_s009", float(test_acc))
        trial.set_user_attr("test_loss_s009", float(test_loss))

        # Now evaluate on the full s009 dataset for comparison purposes
        test_loss_full, test_acc_full = model.evaluate(
            [s009_lidar_input, s009_coord_input], s009_y, batch_size=batch_size, verbose=0
        )
        trial.set_user_attr("test_accuracy_s009_full", float(test_acc_full))
        trial.set_user_attr("test_loss_s009_full", float(test_loss_full))

        # ——————————————————————————————— Save history ——————————————————————————————— #
        history_path = os.path.join(history_dir, f"trial_{trial.number}.csv")

        # Create a DataFrame with all history data
        history_data = {
            "epoch": list(range(1, len(history.history["loss"]) + 1)),
            "train_loss": history.history["loss"],
            "val_loss": history.history["val_loss"],
        }

        # Add accuracy metrics if available
        if "accuracy" in history.history:
            history_data["train_accuracy"] = history.history["accuracy"]
        if "val_accuracy" in history.history:
            history_data["val_accuracy"] = history.history["val_accuracy"]

        # Convert to DataFrame and save as CSV
        history_df = pd.DataFrame(history_data)
        history_df.to_csv(history_path, index=False)

        # ————————————————————————— Finish the current trial ————————————————————————— #
        if len(loss_values) > 1:  # Termination Judgement Report
            report_cross_validation_scores(trial, scores=loss_values)

        return loss  # Value to minimize or maximize

    except optuna.exceptions.TrialPruned:
        raise  # simply propagate pruning
    except tf.errors.ResourceExhaustedError as oom_err:
        print(f"\n❌ Trial {trial.number} hit OOM (Resource Exhausted)\n")
        with open(os.path.join(logs_dir, f"oom_trials.log"), "a") as f:
            f.write(f"OOM error during trial {trial.number}:\n{traceback.format_exc()}\n\n")
        return float("inf") if DIRECTION == "minimize" else float("-inf")
    except Exception as e:
        with open(os.path.join(logs_dir, f"error_trial_{trial.number}.log"), "w") as f:
            f.write(f"An error occurred during trial:\n{e}\n{traceback.format_exc()}\n\n")
        raise  # Re-raise the exception to propagate it
    finally:
        # Clean up resources
        for v in [
            "model",
            "history",
            "history_df",
        ]:
            if v in globals() and globals()[v] is not None:
                del globals()[v]
        (plt.cla(), plt.clf(), plt.close("all"))

## Main

In [ ]:
try:
    # ———————————————————————————————— Study Setup ——————————————————————————————— #
    # Initialize directories for the study
    (
        study_dir,
        args_dir,
        fig_dir,
        backup_dir,
        history_dir,
        model_dir,
        logs_dir,
    ) = init_study_dirs(RUN_DIR)

    study = optuna.create_study(
        study_name=os.path.basename(study_dir),
        storage=f"sqlite:///{study_dir}/optuna_study.db",
        pruner=optuna.pruners.HyperbandPruner(),
        sampler=(optunahub.load_module(package="samplers/auto_sampler")).AutoSampler(),
        load_if_exists=True,
        direction=DIRECTION,
    )

    improvement_callback = ImprovementStagnationCallback()
    being_pruned_callback = StopIfKeepBeingPruned(threshold=100)
    study.optimize(
        lambda trial: objective(
            trial,
            backup_dir=backup_dir,
            model_dir=model_dir,
            fig_dir=fig_dir,
            logs_dir=logs_dir,
            history_dir=history_dir,
            epochs=EPOCHS,
            size_penalizer=None,
        ),
        n_trials=get_remaining_trials(study, NUM_TRIALS),
        callbacks=[improvement_callback, being_pruned_callback],
        catch=(ValueError, RuntimeError),
        gc_after_trial=True,
        n_jobs=1,  # If you have multiple GPUs/Cores
        show_progress_bar=False,
    )

    # ——————————————————————— Processing the Study Results ——————————————————————— #
    top_trials = get_top_trials(
        study,
        top_k=TOP_K,
        rank_key=RANK_KEY,
        rank_descending=RANK_DESCENDING,
    )

    cleanup_paths = [
        (model_dir, "trial_{trial_id}.keras"),
        (fig_dir, "trial_{trial_id}.png"),
        (history_dir, "trial_{trial_id}.csv"),
    ]

    rename_paths = [
        (model_dir, ".keras"),
        (fig_dir, ".png"),
        (history_dir, ".csv"),
    ]

    extra_attrs = [
        "best_train_accuracy",
        "best_val_accuracy",
        "test_accuracy_s009",
        "test_accuracy_s009_full",
        "test_loss_s009",
        "test_loss_s009_full",
    ]

    save_top_k_trials(
        top_trials,
        args_dir=args_dir,
        study=study,
        extra_attrs=extra_attrs,
    )
    cleanup_non_top_trials(
        {t.number for t in study.trials},  # All trials
        {t.number for t in top_trials},  # Top trials ids
        cleanup_paths,
    )
    rename_top_k_files(top_trials, rename_paths)

    # ————————————————————————————— Log Trial Results ———————————————————————————— #
    with open(f"{study_dir}/trials.log", "w") as f:
        f.write(
            f"Total trials: {len(study.trials)}\n"
            f"Pruned trials: {sum(t.state==TrialState.PRUNED for t in study.trials)}\n"
            f"Failed trials: {sum(t.state==TrialState.FAIL for t in study.trials)}\n"
        )

    # —————————————————————————— Generate Study Analysis ————————————————————————— #
    (clear(), analyze_study(study, table_dir=os.path.join(study_dir, "analysis")))

except Exception as e:
    print(f"\n An error occurred: {e}\n")
    traceback.print_exc()

    with open(os.path.join(logs_dir, "training_error.log"), "a") as f:
        f.write(f"An error occurred during training:\n{e}\n{traceback.format_exc()}\n\n")
finally:
    # Write success flag for the auto restart script
    Path("/tmp/success.flag").write_text("SUCCESS")

    # Clean up directories
    shutil.rmtree(backup_dir, ignore_errors=True)
    if not os.listdir(logs_dir):
        os.rmdir(logs_dir)

[I 2025-07-11 08:56:24,018] Using an existing study with name 'optuna_study' instead of creating a new one.
I0000 00:00:1752234984.563539 2180601 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 4756 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3070 Ti, pci bus id: 0000:b3:00.0, compute capability: 8.6


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ lidar_input         │ (None, 20, 200,   │          0 │ -                 │
│ (InputLayer)        │ 10)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lidar_transform_to… │ (None, 20, 200,   │          0 │ lidar_input[0][0] │
│ (Lambda)            │ 4)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ coord_input         │ (None, 2)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lidar_flatten_4_ch… │ (None, 4000, 4)   │          0 │ lidar_transform_… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ coord_tile_flat     │ (None, 4000, 2)   │          0 │ coord_input[0][0] │
│ (Lambda)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ combine_lidar_coord │ (None, 4000, 6)   │          0 │ lidar_flatten_4_… │
│ (Concatenate)       │                   │            │ coord_tile_flat[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_0 (Conv1D)   │ (None, 4000, 512) │     12,288 │ combine_lidar_co… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_0_bn         │ (None, 4000, 512) │      2,048 │ conv1d_0[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_0_act        │ (None, 4000, 512) │          0 │ conv1d_0_bn[0][0] │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pool_0          │ (None, 500, 512)  │          0 │ conv1d_0_act[0][… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ se_0_se_squeeze     │ (None, 512)       │          0 │ max_pool_0[0][0]  │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ se_0_se_reduce      │ (None, 32)        │     16,416 │ se_0_se_squeeze[… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ se_0_se_expand      │ (None, 512)       │     16,896 │ se_0_se_reduce[0… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ se_0_se_reshape     │ (None, 1, 512)    │          0 │ se_0_se_expand[0… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ se_0_se_scale       │ (None, 500, 512)  │          0 │ max_pool_0[0][0], │
│ (Multiply)          │                   │            │ se_0_se_reshape[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_cnn_output  │ (None, 256000)    │          0 │ se_0_se_scale[0]… │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_0 (Dense)     │ (None, 450)       │ 115,200,4… │ flatten_cnn_outp… │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 115,531,154 (440.72 MB)

 Trainable params: 115,530,130 (440.71 MB)

 Non-trainable params: 1,024 (4.00 KB)

Epoch 1/100


I0000 00:00:1752234987.250961 2180775 cuda_dnn.cc:529] Loaded cuDNN version 90501
